<a href="https://colab.research.google.com/github/Haruki-24/Challenge-Alura-Agente-OCI/blob/main/Challenge_Alura_Agente_RAG_V1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto: Agente de Operaciones y Control Industrial (OCI)

## 0 - Configuracion de entorno

### Instalación de depedencias

In [1]:
# 1. Creamos el archivo requirements.txt incluyendo el paquete faltante
requirements_content = """
faiss-cpu
google-generativeai
langchain
langchain-community
langchain-core
langchain-google-genai
langgraph
openpyxl
pandas
PyMuPDF
pypdf
python-dotenv
unstructured
langchain-experimental
langchain-huggingface
sentence-transformers
transformers
"""

with open("requirements.txt", "w") as f:
    f.write(requirements_content.strip())

print("Archivo 'requirements.txt' actualizado actualizado con éxito.")

# 2. Instalamos de manera limpia
!pip install -q -r requirements.txt
!pip install -q "unstructured[pdf]"

print("¡Entorno configurado con éxito!")

Archivo 'requirements.txt' actualizado con langchain-huggingface.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 7.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

### Importación de biibliotecas y conexión con LLMs

In [2]:
import os
from getpass import getpass
import pandas as pd
# Importaciones clave de LangChain / Google GenAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS

print("Bibliotecas base importadas con éxito.")

Bibliotecas base importadas con éxito.


/tmp/ipykernel_883/1331928556.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
# Recuperacion y validacion de Api key
from google.colab import userdata
GEMINI_API_KEY = userdata.get('Gemini-GC')

In [ ]:

# Inicialización del LLM para tareas de análisis y síntesis
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-preview-09-2025", temperature=0.1)
print("Gemini 2.5 configurado correctamente.")


## 1 - Extracción y carga de datos

In [4]:
!pip install -q "unstructured[pdf]" # 1. Asegurar la instalación de las dependencias necesarias

!pip show unstructured # 2. Verificación de la instalación

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.2/542.2 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 16.5 MB/s eta 0:00:00
Name: unstructured
Version: 0.24.0
Summary: A library that prepares raw documents for downstream ML tasks.
Home-page: https://github.co

In [ ]:
# Importaciones necesarias para el cargador fusionado
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, UnstructuredExcelLoader
from langchain_community.document_loaders.merge import MergedDataLoader

# 1. Definición de la ruta de documentos
url_docs = '/content/drive/MyDrive/Colab Notebooks/Challenge Alura 2026/Challenge-Alura-Agente-OCI/data'

# 2. Configuración de cargadores
loader_pdfs = DirectoryLoader(url_docs, glob="*.pdf", loader_cls=PyPDFLoader)
loader_excel = DirectoryLoader(url_docs, glob="*.xlsx", loader_cls=UnstructuredExcelLoader)

# 3.Combinacion de Loaders
all_loaders = MergedDataLoader(loaders=[loader_pdfs, loader_excel])

print("⏳ Cargando documentos desde Drive...")
try:
    documentos_oci = all_loaders.load()
    print(f"Se han cargado exitosamente {len(documentos_oci)} documentos en total.")
    if len(documentos_oci) > 0:
        print(f"Primer documento cargado: {documentos_oci[0].metadata.get('source', 'Desconocido')}")
except Exception as e:
    print(f"Error al cargar los archivos: {e}")
    print("Verifica que tu Google Drive esté montado correctamente con: from google.colab import drive; drive.mount('/content/drive')")


In [8]:
# documentos_oci


## 2 - Fragmentación

In [16]:
from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Descargamos el tokenizador
model_id = 'intfloat/multilingual-e5-small'
print(f"Descargando tokenizador para '{model_id}'...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 2. Configuramos el splitter para (400 tokens rango óptimo)
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=400,    # Límite seguro para dar margen a metadatos (maximo para modelo 512)
    chunk_overlap=50   # Margen de solapamiento semántico
)

# 3. Ejecutamos la fragmentación de los documentos cargados
documentos_fragmentados = text_splitter.split_documents(documentos_oci)

# 4. Inyección del prefijo "passage: " requerida por E5-small para almacenamiento técnico
for doc in documentos_fragmentados:
    if not doc.page_content.startswith("passage: "):
        doc.page_content = f"passage: {doc.page_content}"

print(f"\nFragmentación completada con éxito.")
print(f"Documentos originales: {len(documentos_oci)}")
print(f"Fragmentos (chunks) resultantes listos para indexar: {len(documentos_fragmentados)}")

# Muestra del primer fragmento formateado
if len(documentos_fragmentados) > 0:
    print("\n--- Ejemplo de fragmento formateado para E5 ---")
    print(documentos_fragmentados[0].page_content[:300] + "...")


Descargando tokenizador para intfloat/multilingual-e5-small...

Fragmentación completada con el Tokenizer de Hugging Face.
-> Total de fragmentos generados para indexar: 11

--- Muestra del primer fragmento optimizado para E5 ---
passage: Política de Salud, Seguridad y Medio Ambiente (HSE) - OCI
1. Compromiso Fundamental
En Operations & Control Industrial (OCI), reconocemos que la integridad de nuestro personal, la protección dela salud y el cuidado del medio ambiente son val...


## 3 - Vectorización (Embedings)

In [17]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Configuración de parámetros de Embedding locales
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': True}

print("Inicializando modelo de Embeddings en memoria local...")
embeddings_local = HuggingFaceEmbeddings(
    model_name=model_id,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

# 2. Ruta de persistencia local en Colab
FAISS_DB_PATH = "/content/faiss_index_oci"

print("Generando base de datos vectorial FAISS...")
try:
    # Creamos el índice utilizando los documentos_fragmentados de la Celda 4
    db = FAISS.from_documents(documentos_fragmentados, embeddings_local)

    # Guardamos localmente el índice estructurado
    db.save_local(FAISS_DB_PATH)
    print(f"¡Base de datos FAISS guardada con éxito en '{FAISS_DB_PATH}'!")
except Exception as e:
    print(f"Error durante el proceso de indexación: {e}")

Cargando modelo de embeddings local (multilingual-e5-small)...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Generando vectores e indexando en base de datos FAISS local...
¡Base de datos FAISS creada y guardada con éxito en: '/content/faiss_index_oci'!


## 4 -Recuperacion RAG

In [18]:
def test_de_recuperacion_e5(query, k=3):
    """
    Realiza una búsqueda semántica usando el formato asimétrico de E5 ("query: ").
    Retorna una lista de tuplas (Documento, Score) para que el pipeline del LLM pueda usarlo.
    """
    # Formateamos obligatoriamente la consulta para el modelo E5
    query_formateada = f"query: {query}"

    print(f"\n🔍 Buscando contexto técnico para: '{query}'")
    print(f"🤖 (Consulta formateada enviada a FAISS: '{query_formateada}')")
    print("-" * 75)

    try:
        # Búsqueda por similitud coseno
        resultados = db.similarity_search_with_relevance_scores(query_formateada, k=k)
    except NameError:
        # En caso de que se haya reiniciado el entorno, intentamos cargar el índice desde disco
        print("⚠️ Variable 'db' no encontrada en memoria. Cargando índice guardado...")
        global db
        db = FAISS.load_local(FAISS_DB_PATH, embeddings_local, allow_dangerous_deserialization=True)
        resultados = db.similarity_search_with_relevance_scores(query_formateada, k=k)

    if not resultados:
        print("❌ No se encontraron coincidencias en la base de datos.")
        return []

    for idx, (doc, score) in enumerate(resultados, 1):
        origen = doc.metadata.get('source', 'Desconocido').split('/')[-1]
        print(f"[{idx}] Coincidencia (Relevancia: {score:.4f})")
        print(f"    Fuente original: {origen}")

        # Removemos temporalmente "passage: " solo para facilitar la visualización en pantalla
        contenido_limpio = doc.page_content.replace("passage: ", "").replace('\n', ' ')[:200]
        print(f"    Contenido: {contenido_limpio}...\n")

    return resultados


In [ ]:
# --- PRUEBA DE FUNCIONAMIENTO ---
# Probamos con la consulta operativa real
contexto_recuperado = test_de_recuperacion_e5(
    query="tengo que realizar mantenimiento electrico en campo, ¿que debo llevar?",
    k=4
)

### 1. Transformación de la pregunta en embedding

In [ ]:
# Función de búsqueda semántica
# Implementación de filtros por metadatos (ej. filtrar solo por "Pozo PCP")
# Componente de Reranking (opcional, para mejorar precisión)

###  2. Búsqueda semántica en la base de datos vectorial

### 3. Filtrado por metadados

### 4. Reclasificación (Reranking)

### 5. Ensamblaje del contexto

## 5 - Producción y validación de respuestas

In [ ]:
# Prompt Engineering (Definiendo el rol del Agente OCI y la regla "Human-in-the-loop")
# Lógica de fallback: "No cuento con esa información en los manuales actuales..."
# Función final de la aplicación

### 1. Generación de la respuesta

### 2. Citación de la fuente

### 3. Validación y control de alucinación

### 4. Fallback (alternativa) cuando no hay respuesta

### 5. Formato final